In [4]:
# 3rd party imports
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local imports
sys.path.append("../")
from utils.logs import make_logger
from data_processing import TransportationDataProcessor

# Set up logging
logger = make_logger(include_stdout=True, log_prefix="test")

# Set up the processor
processor = TransportationDataProcessor(logger=logger)

# Run the preprocessing steps
processor.preprocess_data()

print("Static data:\n", processor.static_data.head())
print("Dynamic data:\n", processor.dynamic_data.head())

# Construct the GNN data
gnn_data = processor.prepare_gnn_data()
for split, graphs in gnn_data.items():
    if graphs:
        print(f"{split}: {len(graphs)} graphs")
        print(
            f"  Sample graph - Nodes: {graphs[0].x.shape[0]}, "
            f"Features: {graphs[0].x.shape[1]}, "
            f"Edges: {graphs[0].edge_index.shape[1]}"
        )


/Users/robstallman/projects/transportation-models/scripts/data_processing.py:231: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.2 0.2 1.  0.  0.4]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.static_data.loc[:, static_features_to_scale.columns] = (
/Users/robstallman/projects/transportation-models/scripts/data_processing.py:231: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.         0.66666667 0.16666667 1.         0.        ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.static_data.loc[:, static_features_to_scale.columns] = (
/Users/robstallman/projects/transportation-models/scripts/data_processing.py:231: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.68181818 1.   

Static data:
   link_id from_node_id to_node_id Station ID    Abs PM    Length  \
0     0.0          0.0        1.0        0.0  0.044352  1.000000   
1     1.0          1.0        2.0        1.0  1.000000  0.829787   
2     2.0          2.0        3.0        2.0  0.686576  0.031915   
3     3.0          3.0        4.0        3.0  0.813128  0.000000   
4     4.0          4.0        5.0        4.0  0.000000  0.000000   

   free_flow_speed  congestion_wave_speed  jam_density  
0              0.2               0.000000     0.681818  
1              0.2               0.666667     1.000000  
2              1.0               0.166667     0.000000  
3              0.0               1.000000     0.318182  
4              0.4               0.000000     0.045455  
Dynamic data:
             timestamp Station ID  pct_observed  normalized_flow_[vphpl]  \
0 2022-01-01 00:00:00        0.0      0.683318                 0.151394   
1 2022-01-01 00:05:00        0.0      0.173247                 0.16467

In [117]:
processor.static_data

,link_id,from_node_id,to_node_id,Station ID,Abs PM,Length,free_flow_speed,congestion_wave_speed,jam_density
0,0.0,0.0,1.0,0.0,0.044352,1.000000,0.2,0.000000,0.681818
1,1.0,1.0,2.0,1.0,1.000000,0.829787,0.2,0.666667,1.000000
2,2.0,2.0,3.0,2.0,0.686576,0.031915,1.0,0.166667,0.000000
3,3.0,3.0,4.0,3.0,0.813128,0.000000,0.0,1.000000,0.318182
4,4.0,4.0,5.0,4.0,0.000000,0.000000,0.4,0.000000,0.045455


In [118]:
processor.dynamic_data

,timestamp,Station ID,pct_observed,normalized_flow_[vphpl],normalized_density_[vpmpl]
0,2022-01-01 00:00:00,0.0,0.683318,0.151394,0.210101
1,2022-01-01 00:05:00,0.0,0.173247,0.164675,0.229814
2,2022-01-01 00:10:00,0.0,0.207836,0.151394,0.198663
3,2022-01-01 00:15:00,0.0,0.842392,0.162019,0.156912
4,2022-01-01 00:20:00,0.0,0.326475,0.156707,0.184998
...,...,...,...,...,...
10075,2022-01-07 23:35:00,4.0,0.864393,0.083898,0.107117
10076,2022-01-07 23:40:00,4.0,0.685537,0.080085,0.103500
10077,2022-01-07 23:45:00,4.0,0.759967,0.088983,0.120783
10078,2022-01-07 23:50:00,4.0,0.563372,0.062288,0.138341


In [114]:
from torch_geometric.loader import DataLoader as GNNDataLoader
batch_size = 32
gnn_train_loader = GNNDataLoader(gnn_data["train"], batch_size=batch_size, shuffle=True, drop_last=True)
gnn_val_loader = GNNDataLoader(gnn_data['val'], batch_size=batch_size, shuffle=False)
gnn_test_loader = GNNDataLoader(gnn_data['test'], batch_size=batch_size, shuffle=False)

for batch in gnn_train_loader:
    print(batch)
    print(batch.num_graphs)

DataBatch(x=[160, 7], edge_index=[2, 128], edge_attr=[128, 1], y=[160], timestamp=[32], batch=[160], ptr=[33])
32
DataBatch(x=[160, 7], edge_index=[2, 128], edge_attr=[128, 1], y=[160], timestamp=[32], batch=[160], ptr=[33])
32
DataBatch(x=[160, 7], edge_index=[2, 128], edge_attr=[128, 1], y=[160], timestamp=[32], batch=[160], ptr=[33])
32
DataBatch(x=[160, 7], edge_index=[2, 128], edge_attr=[128, 1], y=[160], timestamp=[32], batch=[160], ptr=[33])
32
DataBatch(x=[160, 7], edge_index=[2, 128], edge_attr=[128, 1], y=[160], timestamp=[32], batch=[160], ptr=[33])
32
DataBatch(x=[160, 7], edge_index=[2, 128], edge_attr=[128, 1], y=[160], timestamp=[32], batch=[160], ptr=[33])
32
DataBatch(x=[160, 7], edge_index=[2, 128], edge_attr=[128, 1], y=[160], timestamp=[32], batch=[160], ptr=[33])
32
DataBatch(x=[160, 7], edge_index=[2, 128], edge_attr=[128, 1], y=[160], timestamp=[32], batch=[160], ptr=[33])
32
DataBatch(x=[160, 7], edge_index=[2, 128], edge_attr=[128, 1], y=[160], timestamp=[32], 

In [111]:
# Get the first graph object
data = gnn_data['train'][0]

print(data)
print('===========================================================================================================')

# Gather some statistics about the graph.
print(f'Number of nodes: {data.num_nodes}')
print(f'Number of edges: {data.num_edges}')
print(f'Average node degree: {data.num_edges / data.num_nodes:.2f}')
print(f'Has isolated nodes: {data.has_isolated_nodes()}')
print(f'Has self-loops: {data.has_self_loops()}')
print(f'Is undirected: {data.is_undirected()}')

Data(x=[5, 7], edge_index=[2, 4], edge_attr=[4, 1], y=[5], timestamp=2022-01-01 00:00:00)
Number of nodes: 5
Number of edges: 4
Average node degree: 0.80
Has isolated nodes: False
Has self-loops: False
Is undirected: False


In [90]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, num_features: int, num_targets: int, num_hidden_layers: int = 16):
        super().__init__()
        self.conv1 = GCNConv(num_features, num_hidden_layers)
        self.conv2 = GCNConv(num_hidden_layers, num_targets)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.log_softmax(x, dim=1)

        return x.mean(dim=1)

In [116]:
device = torch.device('cpu')
model = GCN(num_features=7, num_targets=160).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    for batch, data in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(data)
        loss = loss_fn(pred, data.y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 10 == 0:
            loss, current = loss.item(), batch * batch_size + len(data.x)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for data in dataloader:
            pred = model(data)
            test_loss += loss_fn(pred, data.y).item()
            correct += (pred.argmax(1) == data.y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
         
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5)
epochs = 100
for t in range(epochs):
    print(f"Epoch {t+1}\n----------------------------------------")
    train_loop(gnn_train_loader, model, loss_fn, optimizer)


Epoch 1
----------------------------------------
loss: 26.953003  [  160/ 1411]
loss: 26.936865  [  480/ 1411]
loss: 26.942245  [  800/ 1411]
loss: 26.964581  [ 1120/ 1411]
loss: 26.930243  [ 1440/ 1411]
Epoch 2
----------------------------------------
loss: 26.924999  [  160/ 1411]
loss: 26.939758  [  480/ 1411]
loss: 26.934280  [  800/ 1411]
loss: 26.942596  [ 1120/ 1411]
loss: 26.960016  [ 1440/ 1411]
Epoch 3
----------------------------------------
loss: 26.947052  [  160/ 1411]
loss: 26.960598  [  480/ 1411]
loss: 26.945972  [  800/ 1411]
loss: 26.938391  [ 1120/ 1411]
loss: 26.959635  [ 1440/ 1411]
Epoch 4
----------------------------------------
loss: 26.920689  [  160/ 1411]
loss: 26.946198  [  480/ 1411]
loss: 26.926189  [  800/ 1411]
loss: 26.954128  [ 1120/ 1411]
loss: 26.952753  [ 1440/ 1411]
Epoch 5
----------------------------------------
loss: 26.958765  [  160/ 1411]
loss: 26.948044  [  480/ 1411]
loss: 26.930573  [  800/ 1411]
loss: 26.966801  [ 1120/ 1411]
loss: 26.95